Paquetes necesarios

In [ ]:
import cv2  
import math 
import csv
from ultralytics import YOLO

In [ ]:
# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') # Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car"]

#Para un vídeo 
filename = "assets/C0142.mp4"
results = model(source=filename,
                show=True,
                imgsz=320)

cv2.destroyAllWindows()


In [ ]:
# Modelo RTDETR

In [ ]:
# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') # Contenedores
model2 = "El creado por nosotros"

detect = set()

clases_vehiculos = [2, 3, 5, 7]

cap = cv2.VideoCapture("assets/C0142.mp4")

total_clases = {
    0: set(),  # person
    2: set(),  # coches
    3: set(),  # motos
    5: set(),  # trucks
    7: set()   # buses
}

ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

output_video = cv2.VideoWriter('output_video.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto))

frame_count = 0

csv_file = open('detections.csv', mode='w', newline='')
writer = csv.writer(csv_file)
writer.writerow(['frame', 'class', 'confidence', 'track_id', 'x1', 'y1', 'x2', 'y2', 'matrícula_en_su_caso', 'confianza', 'mx1', 'my1', 'mx2', 'my2', 'texto_matricula'])


while True:
    ret, frame = cap.read()
    frame_count += 1
    
    if not ret:
        break

    # Para cada detección
    results = model.track(source=frame, persist=True, imgsz=320, verbose=False)
    
    for box in results[0].boxes:
        # Contenedor
        x1, y1, x2, y2 = box.xyxy[0]
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) 

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Clase
        cls = int(box.cls[0])

        # Parametros Csv
        conf_model = float(box.conf[0])
        objeto = model.names[cls]

        track_id = -1  # Valor por defecto si no hay ID de seguimiento
        if box.id is not None:
            track_id = int(box.id[0])

            if cls in total_clases:
                total_clases[cls].add(track_id)


        tiene_mat = 0
        conf_mat = 0.0
        g_x1, g_y1, g_x2, g_y2 = 0, 0, 0, 0
        texto_matricula = ""

        if cls in clases_vehiculos:
            vehiculo_recortado = frame[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
            
            results2 = model2(source=vehiculo_recortado, imgsz=320)


            if len(results2[0].boxes) > 0:
                box2 = results2[0].boxes[0]
                tiene_mat = 1
                conf_mat = float(box2.conf[0])

                x1_2, y1_2, x2_2, y2_2 = box2.xyxy[0]
                x1_2, y1_2, x2_2, y2_2 = int(x1_2), int(y1_2), int(x2_2), int(y2_2)

                g_x1, g_y1 = x1 + x1_2, y1 + y1_2
                g_x2, g_y2 = x1 + x2_2, y1 + y2_2

                cv2.rectangle(frame, (g_x1, g_y1), (g_x2, g_y2), (255, 0, 0), 2)

        writer.writerow([
            frame_count, objeto, round(conf_model, 2), track_id,
            x1, y1, x2, y2,
            tiene_mat, round(conf_mat, 2), g_x1, g_y1, g_x2, g_y2, texto_matricula
        ])
    output_video.write(frame)

    cv2.imshow("Detección de vehículos", frame)
    if cv2.waitKey(20) == 27:
        break

for cls, boxes in total_clases.items():
    print(f"Clase {cls}: {len(boxes)} detecciones")

cv2.destroyAllWindows()
output_video.release()
cap.release()
csv_file.close()

NameError: name 'YOLO' is not defined